## License
Copyright 2026 jphall@gwu.edu. MIT License; see the repository LICENSE file.

# LLM as judge

This notebook generates two answers, then sends each answer to a separate LLM evaluator with a transparent rubric. 

The rubric adapts slide 24 of Lecture 2: Transformer-based LLM: Accuracy, Relevance, Completeness, Clarity, Instruction Following, Usefulness, and Safety/Appropriateness. Every dimension is scored 1 to 3.


## 1. Import packages and configure Azure


In [ ]:
# 1. Import packages and configure Azure
# Load table and JSON helpers before creating the Azure client.

import json
import pandas as pd
import os
from getpass import getpass
from openai import AzureOpenAI

RESOURCE = "gw-sb-01"
ENDPOINT = f"https://{RESOURCE}.openai.azure.com/"

api_key = os.getenv("GW_AZURE_OPENAI_KEY") or getpass("Azure OpenAI API key: ")
client = AzureOpenAI(azure_endpoint=ENDPOINT, api_key=api_key, api_version="2025-03-01-preview")


## 2. Define questions and the rubric


In [ ]:
# 2. Define questions and the rubric
# Make every evaluation dimension visible before asking the judge.

queries = ["Solve 17 * 24, then explain briefly.", "What are the most fundamental differences between the outlooks of Derrida and Foucault?"]
rubric = {"Accuracy": "factually correct", "Relevance": "directly responsive", "Completeness": "covers key parts", "Clarity": "clear and organized", "Instruction Following": "follows the request", "Usefulness": "practical and helpful", "Safety/Appropriateness": "safe and appropriate"}

rubric


{'Accuracy': 'factually correct',
 'Relevance': 'directly responsive',
 'Completeness': 'covers key parts',
 'Clarity': 'clear and organized',
 'Instruction Following': 'follows the request',
 'Usefulness': 'practical and helpful',
 'Safety/Appropriateness': 'safe and appropriate'}

## 3. Generate answers and collect ratings


In [ ]:
# 3. Generate answers and collect ratings
# Use a separate model call for each answer and display structured results.

# This schema guarantees a predictable object for the table below.
# It replaces unreliable prompt-only instructions such as "return JSON".
rating_names = list(rubric)

# The nested ratings object has one required 1-3 score for every rubric item.
judge_schema = {
    "type": "object",
    "properties": {
        "ratings": {
            "type": "object",
            "properties": {name: {"type": "integer", "minimum": 1, "maximum": 3} for name in rating_names},
            "required": rating_names,
            "additionalProperties": False,
        },
        "total": {"type": "integer", "minimum": 7, "maximum": 21},
        "feedback": {"type": "string"},
    },
    "required": ["ratings", "total", "feedback"],
    "additionalProperties": False,
}

def generate_answer(question):
    # Leave room for GPT-5 internal reasoning and a visible answer.
    response = client.responses.create(
        model="gpt-5-mini",
        input=question,
        max_output_tokens=1600,
        reasoning={"effort": "minimal"},
    )
    answer = response.output_text.strip()

    # Stop here rather than asking the judge to score an empty answer.
    if not answer:
        raise RuntimeError("The answer model returned no visible text. Please run this question again.")

    return answer

def evaluate(question):
    answer = generate_answer(question)

    # Give the separate judge both the original question and the generated answer.
    prompt = f"Score the answer from 1 to 3 on each rubric dimension. Rubric: {json.dumps(rubric)} Question: {question} Answer: {answer}"
    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt,
        # The JSON rubric needs extra visible tokens after internal reasoning.
        max_output_tokens=1200,
        reasoning={"effort": "minimal"},
        text={
            "format": {
                "type": "json_schema",
                "name": "answer_evaluation",
                "strict": True,
                "schema": judge_schema,
            }
        },
    )

    if not response.output_text:
        raise RuntimeError("The judge returned no visible evaluation. Please run the cell again.")

    # Convert the schema-constrained JSON text into a Python dictionary.
    return answer, json.loads(response.output_text)

rows = []

# Evaluate each answer, then make one easy-to-read table row per question.
for question in queries:
    answer, judgment = evaluate(question)
    rows.append({"question": question, "answer": answer, **judgment["ratings"], "total": judgment["total"], "feedback": judgment["feedback"]})

pd.DataFrame(rows)


,question,answer,Accuracy,Relevance,Completeness,Clarity,Instruction Following,Usefulness,Safety/Appropriateness,total,feedback
0,"Solve 17 * 24, then explain briefly.",17 × 24 = 408.\n\nBrief explanation: 24 × 10 =...,3,3,3,3,3,3,3,21,The answer is correct and directly addresses t...
1,What are the most fundamental differences betw...,Here’s a concise comparison of the most fundam...,3,3,3,3,3,3,3,21,The answer accurately and succinctly distingui...
